In [15]:
import requests
from bs4 import BeautifulSoup

url = "https://stripe.com/docs/api"
response = requests.get(url)
soup = BeautifulSoup(response.text, 'html.parser')
text = soup.get_text()
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import chromadb
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import requests
from bs4 import BeautifulSoup


In [16]:

device = "cuda"
base_model_id = "Qwen/Qwen2.5-1.5B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    device_map="auto",
    torch_dtype=torch.float16
)
tokenizer = AutoTokenizer.from_pretrained(base_model_id)

def generate_with_qwen(messages):
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer([text], return_tensors="pt").to(device)
    output = model.generate(
        inputs.input_ids,
        max_new_tokens=512,
        do_sample=False,
        temperature=None,
        top_p=None,
        top_k=None
    )
    output = output[0][len(inputs.input_ids[0]):]
    return tokenizer.decode(output, skip_special_tokens=True)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [5]:

pages = [
    "https://stripe.com/docs/api",
    "https://stripe.com/docs/payments",
    "https://stripe.com/docs/billing",
    "https://stripe.com/docs/webhooks",
    "https://docs.stripe.com/api/authentication",
    "https://docs.stripe.com/api/errors",
    "https://docs.stripe.com/api/include_dependent_response_values"
]

docs = []

for url in pages:
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')
    
    for script in soup(["script", "style", "nav"]):
        script.decompose()
    
    text = soup.get_text()
    
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    clean_text = "\n".join(lines)
    
    docs.append({
        "url": url,
        "text": clean_text
    })
    print(f"{url} — {len(clean_text)} letter")


https://stripe.com/docs/api — 10154 letter
https://stripe.com/docs/payments — 3303 letter
https://stripe.com/docs/billing — 3909 letter
https://stripe.com/docs/webhooks — 28231 letter
https://docs.stripe.com/api/authentication — 11092 letter
https://docs.stripe.com/api/errors — 13003 letter
https://docs.stripe.com/api/include_dependent_response_values — 13717 letter


In [6]:
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

chunks=[]
for doc in docs:
    splits=splitter.split_text(doc['text'])
    for split in splits:
        chunks.append({"text": split,"source": doc['url']})
print(len(chunks))

199


In [7]:
client = chromadb.PersistentClient(path="/home/hisham/stripe-rag/chroma_db")
collection = client.get_or_create_collection("stripe_docs")

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

texts=[chunk['text'] for chunk in chunks]
sources=[chunk['source'] for chunk in chunks]
ids=[str(i) for i in range(len(chunks))]

embeddings=embedding_model.encode(texts).tolist()
collection.add(documents=texts,
               embeddings=embeddings,
               metadatas=[{"source": s} for s in sources],
               ids=ids)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:
print(collection.count())

199


In [17]:
question = "How do webhooks work?"

question_embedding = embedding_model.encode([question]).tolist()

results = collection.query(
    query_embeddings=question_embedding,
    n_results=3
)

for i, doc in enumerate(results['documents'][0]):
    print(f"--- result {i+1} ---")
    print(doc[:300])
    print()

--- result 1 ---
up an HTTP or HTTPS endpoint function that can accept webhook requests with a POST method. If you’re still developing your endpoint function on your local machine, it can use HTTP. After it’s publicly accessible, your webhook endpoint function must use HTTPS.Set up your endpoint function so that it:

--- result 2 ---
an organization webhook endpointIf you create a webhook endpoint in an organization account, select Accounts to listen to events from accounts in your organization. If you have Connect platforms as members of your organizations and want to listen to events from the all the platforms’ connected accou

--- result 3 ---
server and we don’t recommend it.You can change the events that a webhook endpoint receives in the Dashboard or with the API.Handle events asynchronouslyConfigure your handler to process incoming events with an asynchronous queue. You might encounter scalability issues if you choose to process event



In [18]:
context = "\n\n".join(results['documents'][0])

messages = [
    {
        "role": "system",
        "content": "You are a helpful assistant that answers questions based on Stripe documentation. Answer only from the provided context."
    },
    {
        "role": "user",
        "content": f"""Context:
{context}

Question: {question}

Answer:"""
    }
]
response = generate_with_qwen(messages)
print(response)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Webhooks work by allowing applications to receive notifications about specific events occurring within their systems without having to actively monitor those events themselves. When an event occurs, such as a payment being made or a subscription renewal, the system hosting the application sends a notification to a specified URL, known as the webhook endpoint.

This is typically done using HTTP or HTTPS protocols, depending on whether the application is running locally or on a public server. The notification includes information about the event, often in a structured format like JSON, along with metadata about the source of the event.

For instance, if an order is placed on an e-commerce platform, the platform would send a webhook to the merchant's website, notifying them of the new order. This allows the merchant to update their database, notify customers, or perform other actions related to the new order without needing to continuously check for orders themselves.

The key components 

In [ ]:
def chat(question):
    question_embedding = embedding_model.encode([question]).tolist()
    
    results = collection.query(
        query_embeddings=question_embedding,
        n_results=3
    )
    
    context = "\n\n".join(results['documents'][0])
    sources = [m['source'] for m in results['metadatas'][0]]
    
    messages = [
        {
            "role": "system",
            "content": "You are a helpful assistant that answers questions based on Stripe documentation. Answer only from the provided context. If the answer is not in the context, say 'I don't know'."
        },
        {
            "role": "user",
            "content": f"""Context:
{context}

Question: {question}

Answer:"""
        }
    ]
    
    response = generate_with_qwen(messages)
    
    print(f"Answer: {response}")
    print(f"\nSources:")
    for source in set(sources):
        print(f"  - {source}")

while True:
    question = input("\nAsk a question (or 'quit' to exit): ")
    if question.lower() == 'quit':
        break
    chat(question)